# Qwen3-32B chat on OOI

Choose **Run → Run All Cells**. If the model is offline, click **Start GPU** and wait for **Ready**. Enter a message and click **Send**. **New chat** clears the conversation. **Stop GPU** releases the model’s GPU memory and keeps the downloaded files for next time.

This notebook sends your conversation to Qwen3-32B inside your OOI workspace. Supply any ocean measurements or documents you want it to consider.

In [ ]:
import asyncio
import html
import json
import subprocess
import urllib.error
import urllib.request
from pathlib import Path

import ipywidgets as w
from IPython.display import display

MODEL = "Qwen3-32B"
BASE = "http://127.0.0.1:18732"
ROOT = Path.home() / "data/ocean-platform/qwen3-32b"
if (Path.cwd() / "start.sh").is_file():
    ROOT = Path.cwd()
local_http = urllib.request.build_opener(urllib.request.ProxyHandler({}))
history = []
system = {"role": "system", "content": "You are a helpful research assistant. Distinguish provided observations from interpretations. If evidence is missing, say what would help."}

status = w.Label(value="Checking the local model…")
transcript = w.HTML()
prompt = w.Textarea(placeholder="Ask a question, or paste observations to discuss…",
                    layout=w.Layout(width="100%", height="90px"))
send = w.Button(description="Send", button_style="primary")
new = w.Button(description="New chat")
start = w.Button(description="Start GPU")
stop = w.Button(description="Stop GPU")
refresh = w.Button(description="Check status")
busy = False
ready = False


def request(path, data=None, timeout=8):
    headers = {"Content-Type": "application/json"}
    key_path = ROOT / ".api-key"
    if key_path.is_file():
        headers["Authorization"] = "Bearer " + key_path.read_text().strip()
    body = None if data is None else json.dumps(data).encode()
    req = urllib.request.Request(BASE + path, data=body, headers=headers)
    with local_http.open(req, timeout=timeout) as response:
        return json.load(response)


def model_status():
    try:
        request("/health", timeout=3)
        models = request("/v1/models", timeout=3)["data"]
        if any(m.get("id") == MODEL for m in models):
            return True, "Ready · Qwen3-32B"
        return False, "The local endpoint is serving a different model."
    except urllib.error.HTTPError as exc:
        if exc.code == 503:
            return False, "The model is loading. Check status in a moment."
        return False, f"The local server returned HTTP {exc.code}."
    except (OSError, ValueError, KeyError):
        return False, "Offline. Click Start GPU to load the model."


def render():
    blocks = []
    for message in history:
        who = "You" if message["role"] == "user" else "Qwen3-32B"
        value = html.escape(message["content"])
        blocks.append(f"<p><strong>{who}</strong></p><div style='white-space:pre-wrap;overflow-wrap:anywhere'>{value}</div>")
    transcript.value = "".join(blocks)


def controls():
    for button in (new, start, stop, refresh):
        button.disabled = busy
    send.disabled = busy or not ready
    start.disabled = busy or ready
    prompt.disabled = busy


async def check_status():
    global ready
    ready, status.value = await asyncio.to_thread(model_status)


async def service(action):
    global ready
    if action == "start":
        await check_status()
        if ready:
            return
    status.value = "Loading Qwen3-32B…" if action == "start" else "Stopping Qwen3-32B…"
    run = await asyncio.to_thread(subprocess.run, ["bash", str(ROOT / (action + ".sh"))],
                                  cwd=ROOT, capture_output=True, text=True, timeout=240)
    if run.returncode:
        raise RuntimeError(f"{action}.sh returned {run.returncode}. Check the model logs in the notebook folder.")
    if action == "start":
        for _ in range(120):
            await check_status()
            if ready:
                return
            status.value = "Loading Qwen3-32B…"
            await asyncio.sleep(2)
    await check_status()


async def ask():
    question = prompt.value.strip()
    if not question:
        status.value = "Enter a message first."
        return
    user_message = {"role": "user", "content": question}
    payload = {"model": MODEL, "messages": [system, *history, user_message],
               "max_tokens": 768, "temperature": 0.7, "top_p": 0.8,
               "chat_template_kwargs": {"enable_thinking": False}, "stream": False}
    status.value = "Qwen3-32B is generating a reply…"
    response = await asyncio.to_thread(request, "/v1/chat/completions", payload, 240)
    choice = response["choices"][0]
    answer = choice["message"]["content"]
    if not isinstance(answer, str) or not answer.strip():
        raise ValueError("The model returned an empty reply. Retry or start a new chat.")
    history.extend([user_message, {"role": "assistant", "content": answer.strip()}])
    render()
    prompt.value = ""
    status.value = "Ready · reply reached the output limit" if choice.get("finish_reason") == "length" else "Ready · Qwen3-32B"


async def execute(action):
    global busy
    try:
        await action()
    except urllib.error.HTTPError as exc:
        status.value = f"Request failed (HTTP {exc.code}). Check status or shorten the conversation and try again."
    except Exception as exc:
        status.value = str(exc)
    finally:
        busy = False
        controls()


def launch(action):
    global busy
    if busy:
        return
    busy = True
    controls()
    asyncio.create_task(execute(action))


def clear(_):
    history.clear()
    render()
    prompt.value = ""
    status.value = "Ready · new chat" if ready else "New chat. Click Start GPU if the model is offline."


send.on_click(lambda _: launch(ask))
new.on_click(clear)
start.on_click(lambda _: launch(lambda: service("start")))
stop.on_click(lambda _: launch(lambda: service("stop")))
refresh.on_click(lambda _: launch(check_status))
display(w.VBox([w.HBox([start, stop, refresh]), status, transcript, prompt, w.HBox([send, new])]))
launch(check_status)
